# 10 · Pipelines y contrato de entrenamiento

Fase F, paso 31. Consulte [LEEME_FASE_F.md](LEEME_FASE_F.md). Se verifica la ejecución E y se congelan configuración, familias, candidatos, predictores y evaluación. **Modo diagnóstico:** los ajustes ensayan el código con candidatos; no autorizan producción ni validan las etiquetas.

In [1]:
from pathlib import Path
import sys, importlib
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/geoau/training.py').is_file())
if str(ROOT/'src') not in sys.path: sys.path.insert(0, str(ROOT/'src'))
from geoau import training as tr
from geoau import evaluation as ev
importlib.reload(tr)

RUN = tr.ensure_run(ROOT)
display(ev.read_json(RUN/"control_cierre.json"))
display(ev.read_json(RUN/"config_snapshot.json"))

Fase F: C:\Users\Lenovo\Desktop\PROYECTO IA\Proyecto Con Luis\reports\fase_f\20260909T184735_016759Z


{'estado_ejecucion': 'en_curso',
 'mode': 'diagnostic',
 'scientific_training_allowed': False,
 'prediction_allowed': False}

{'schema_version': 1,
 'phase_e_run': 'reports/fase_e/20260909T124128_606717Z',
 'output_directory': 'reports/fase_f',
 'mode': 'diagnostic',
 'allow_diagnostic_fit': True,
 'seed': 42,
 'search_trials': 3,
 'forest_trees': 500,
 'ablation_trees': 150,
 'n_jobs': 2,
 'prediction_batch_size': 20000,
 'evaluation_background_size': 10000,
 'ratios_u_to_p': [3, 1, 10],
 'pu_realizations': [0, 1, 2],
 'fit_weighting': 'none',
 'primary_metric': 'recovery_at_05',
 'feature_set': 'geology_terrain_geo4_hydro',
 'families': ['logistic', 'random_forest', 'extra_trees', 'hist_boosting']}

## Columnas y transformaciones
Solo entran predictores de la lista D. Se excluyen coordenadas, IDs, etiquetas, grupos y pesos. Las categorías y clases geoquímicas se codifican como categorías, con estados explícitos para ausente/desconocido. La mediana, categorías y escalado de regresión logística se aprenden en cada entrenamiento interno. El boosting desactiva la validación aleatoria implícita de early stopping.

In [2]:
schema = ev.read_json(RUN/'feature_schema.json')
display(pd.DataFrame([{'set': name, 'predictors': len(v['columns']), 'categorical': len(v['categorical'])}
                      for name, v in schema.items()]))
display(ev.read_json(RUN/'candidates.json'))

,set,predictors,categorical
0,geology,79,2
1,geology_terrain,85,2
2,geology_terrain_pathfinders,88,5
3,geology_terrain_geo4,89,6
4,geology_terrain_geo4_hydro,93,6
5,geo4_proportions,120,2


{'logistic': [{'candidate_id': 'logistic_00',
   'family': 'logistic',
   'ratio': 3,
   'params': {'C': 1.0}},
  {'candidate_id': 'logistic_01',
   'family': 'logistic',
   'ratio': 1,
   'params': {'C': 0.1}},
  {'candidate_id': 'logistic_02',
   'family': 'logistic',
   'ratio': 10,
   'params': {'C': 10.0}}],
 'random_forest': [{'candidate_id': 'random_forest_00',
   'family': 'random_forest',
   'ratio': 3,
   'params': {'n_estimators': 500,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'max_depth': None}},
  {'candidate_id': 'random_forest_01',
   'family': 'random_forest',
   'ratio': 1,
   'params': {'n_estimators': 500,
    'min_samples_leaf': 20,
    'max_features': 0.5,
    'max_depth': None}},
  {'candidate_id': 'random_forest_02',
   'family': 'random_forest',
   'ratio': 10,
   'params': {'n_estimators': 500,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'max_depth': 16}}],
 'extra_trees': [{'candidate_id': 'extra_trees_00',
   'family': 'extra_t

## Protocolo predefinido
Selección por recuperación de candidatos al priorizar el 5 % del área en validación interna. En modo validado se cuenta una vez cada depósito. Empates por hash de celda independiente de las etiquetas. AP y ROC-AUC P/U se calculan en una muestra fija de evaluación y no miden descubrimiento real. No se generan predicciones de la reserva.

El piloto usa tres candidatos por familia y ratios 3/1/10. La referencia de Random Forest tiene 500 árboles, hoja mínima 5 y `max_features="sqrt"`. La configuración permite ampliar la búsqueda en una nueva ejecución.